# មេរៀនទី 18 (បន្ត): ប័ណ្ណបង្កាន់ដៃ ដែលបង្ហាញថា *មនុស្ស* បានអនុម័តសកម្មភាព

មេរៀននេះបង្ហាញពីអ្វីដែល **ភ្នាក់ងារ** បានធ្វើ និងអ្វីដែល **ទ្វារ** បានសម្រេច។ សៀវភៅកំណត់ត្រានេះបន្ថែមផ្នែកខ្វះគឺ៖ ប្រាក់បបួលថា **មនុស្សមានឈ្មោះ** បានអនុម័តសកម្មភាព **ពេញលេញ** — ជាឥСКSignature** ដាច់ដោយឡែកដែលមនុស្សកាន់ ធ្វើបញ្ជាក់ក្រៅបណ្តាញ។

អត្ថប្រយោជន៍ទាំងពីរនៅទីនេះប្រើ **រូបរាងសំបុត្រដូចសំបុត្រមេរៀន**៖ ទិន្នន័យរលោងជាមួយវាល `type` ចុះហត្ថលេខា Ed25519 លើ SHA-256 នៃប្រៃសណីយ៍ច្រកតាមស្តង់ដារ, មានអ 객체 `signature` ដែលភ្ជាប់ជាផ្នែករចនាសម្ព័ន្ធ (ហើយមិនរាប់បញ្ចូលក្នុងការចុះហត្ថលេខាប្រៃសណីយ៍). ប័ណ្ណអនុម័ត គឺជា `type` ថ្មី (`human.approval.v1`) នៅជាប់ប្រភេទសកម្មភាព, ដូច្នេះ `verify_chain` មួយគ្របដណ្តប់ទាំងពីរប្រភេទអត្ថប្រយោជន៍ជាមួយផ្លូវកូដដូចគ្នាដែលអ្នកបានសរសេរនៅក្នុងសៀវភៅកំណត់ត្រាមេរៀន។ វាក៏ជារូបរាងផ្លូវហត្ថលេខារួមក្នុងកំណត់ត្រាអ៊ីនធឺណិតដែលមេរៀនបានអនុវត្ត (draft-farley-acta-signed-receipts)។

ការកែលម្អមួយដែលបានពិចារណាយ៉ាងម៉ត់ចត់លើកម្មវិធីផ្ទៀងផ្ទាត់នៅក្នុងសៀវភៅកំណត់ត្រាមេរៀន: កម្មវិធីផ្ទៀងផ្ទាត់នៅទីនេះដោះស្រាយ `signature.key_id` ទៅប្រឆាំងនឹង **បញ្ជីកូនសោដាក់ថ្ម** ជំនួសពីការជឿទុកចិត្តកូនសោសាធារណៈដែលបានផ្ទុកនៅក្នុងប័ណ្ណបង្កាន់ដៃ។ នេះគឺជាទ្រឹស្តីផលិតផលដែលបញ្ជីត្រួតពិនិត្យរបស់មេរៀនផ្តល់អនុសាសន៍ ("ផ្សព្វផ្សាយកូនសោសាធារណៈសម្រាប់ផ្ទៀងផ្ទាត់"), ហើយវាជាអ្វីដែលធ្វើឲ្យការបញ្ចូលក្លែងបន្លំជាបដិវិធីមិនទទួលខុសត្រូវមិនមែនជាការចូលប្រើដោយកូនសោផ្ទាល់ខ្លួន។

ច្បាប់ដែលសៀវភៅកំណត់ត្រានេះបង្រៀន៖ **ការអនុម័តដែលបានចុះហត្ថលេខាមិនមែនជាអាជ្ញាធរដោយខ្លួនឯងឡើយ។** អាជ្ញាធរមានតែបើប័ណ្ណអនុម័ត និងប័ណ្ណសកម្មភាពនៅតែភ្ជាប់ទៅនឹងសកម្មភាពតាមបំណងតែមួយនៅពេលអនុវត្ត ភាគីនយោបាយ កូនសោ និងថ្ងៃផុតកំណត់នៅតែទាន់សម័យ ហើយការអនុម័តមិនទាន់បានប្រើប្រាស់ទេ។ ការបរាជ័យរាល់ខ្ទង់ត្រូវបានបដិសេធជាមួយ **ហេតុផលខុសគ្នា** ដូច្នេះអ្នកអាចបំបែកបានរវាង *អាជ្ញាធរបាត់បង់សុពលភាព* និង *សកម្មភាពដែលបានអនុវត្តបានផ្លាស់ប្តូរ*។


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## សកម្មភាព​ដែលបានកំណត់​ត្រឹមត្រូវ

ប្រវត្តិការអនុម័តគឺជា **វត្ថុសកម្មភាពបែប canonical**  — មិនមែនជាពាក្យស្លោក​មិនច្បាស់ដូចជា "អនុម័តការសងប្រាក់វិញ" ទេ ប៉ុន្តែជាសកម្មភាពដ៏ពិតប្រាកដ ដែលបានបញ្ជាក់ពេញលេញ។ ការចុះហត្ថលេខាលើវត្ថុទាំងមូល (ហើយបង្កើតការសង្ខេបពីវា) ជាអ្វីដែលអនុញ្ញាតឲ្យយើងបញ្ជាក់បាននៅពេលក្រោយថាមនុស្សបានអនុម័ត *នេះ* ហើយមិនមែនអ្វីផ្សេងទៀតឡើយ។


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## លិខិតមួយ សិទ្ធិអំណាចពីរគឺមានពីរយោង

រាល់បង្កាន់ដៃគឺជា​លិខិតប្រអប់នៃមេរៀនមួយ: បន្ទុកទំហំប.flat ៗ ជាមួយប្រអប់ `type`  មួយ និងអនុគមន៍ `signature` (`alg`, `sig`, `key_id`) ដែល **មិនមែន**ជាផ្នែកជាសញ្ញាដ៏បានចុះហត្ថលេខានោះទេ។ `verify_envelope` គឺជាការត្រួតពិនិត្យរចនាសម្ព័ន្ធរួម + សញ្ញាជាសញ្ញាដែលទាំងពីរប្រភេទបង្កាន់ដៃ; អ្វីដែលផ្តល់សិទ្ធិម្ដងទៀតយោងទៅលើ `signature.key_id` គឺជា **ការចុះបញ្ជីកូនសោដែលត្រូវបានដាក់ចំណាំ** ដែលធ្វើឲ្យសិទ្ធិអំណាចទាំងពីរនឹងមានការបំបែកធៀបគ្នា:

- **បង្កាន់ដៃអនុម័ត** (`human.approval.v1`) — អ្នកអនុម័តដែលបានបញ្ជាក់ ឯកសារសកម្មភាពពេញលេញដែលជាកាន់តែច្បាស់លាស់ **និងផលបូករបស់វា**, `policy_version`, ពេលវេលាដែលចេញចំណាំ និងកាលបរិច្ឆេទផុតកំណត់។ ការប្រើប្រាស់មួយដងត្រូវបានកត់ត្រានៅកម្រិតខ្សែ។
- **បង្កាន់ដៃសកម្មភាព** (`agent.action.v1`) — អត្តសញ្ញាណតំណាង, `run_id`, ផ្នែកដ៏ស្មើគ្នានៃសកម្មភាព canonical **digest**, លទ្ធផលអនុវត្តន៍ និងពេលវេលា, និង `parent_approval_ref`: គឺជា `receipt_hash` នៃអនុម័តភាព, បទប្បន្ននេះដូចគ្នានឹង `previous_receipt_hash` ក្នុងខ្សែរបស់មេរៀន។

ផ្នែក `action_digest` ដែលចែករួមគឺជាការចងខ្សែលម្អិតដែលការតភ្ជាប់ពឹងផ្អែកលើវា។ `key_id` មាននៅក្នុងអង្គភាពសញ្ញាជាដំណាក់សម្រាប់សរសេរកំណត់តែប៉ុណ្ណោះ: ការផ្លាស់ប្តូរវាទៅកូនសោដែលបានចុះបញ្ជីផ្សេងគ្នាដែលធ្វើឲ្យការត្រួតពិនិត្យសញ្ញាបានបរាជ័យ ដូច្នេះវាមិនបានផ្តល់ៈអ្វីដែលមានតម្លៃឡើយ។


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: ដែល​ការចងក្រង​ត្រូវ​បាន​សម្រេច​នៅ​ពេល​ពិតប្រាកដ

`verify_chain` មិនមែន​ជា​ការវេចខ្ចប់​ងាយស្រួល​លើ​ការ​ពិនិត្យ​ហត្ថលេខា​ពីរ​នោះទេ។ វាជា​កន្លែង​តែមួយ​ដែល​មាន​ការ​ពិនិត្យ​រួម​គ្នា​ពី `action_digest` ដែល​ជា canonical ដែលបានចែកចាយ សេចក្ដីអនុម័ត​លេខាការណ៍/កំណត់​នីតិវិធី/កំណត់​ពេលអនុញ្ញាត ត្រឹមត្រូវ **ថ្មីៗ** និង **ការប្រើប្រាស់​តែមួយ​ដង** នៃការអនុញ្ញាត នេះ​ទាំងអស់​ត្រូវ​បានពិនិត្យ​ជាមួយ​នឹង​សកម្មភាព​ដែល​កំពុង​ធ្វើ​ប្រតិបត្តិ **ឥឡូវនេះ**។

ការបរាជ័យ​ណាមួយ ការពុំយល់ព្រម​នឹង​ផ្តល់​​នូវ​មូលហេតុ​មាន​ភាព​ខុសគ្នា ដូច្នេះ​អ្នក​អាន​សេចក្ដីបដិសេធអាច​យល់​ថា​អំណាច​បាន​ចាស់​បាត់ (នីតិវិធី​បានផ្លាស់ប្ដូរ, គន្លឹះបានបញ្ចុះតម្លៃ, ការអនុញ្ញាតបានផុតកំណត់, ការអនុញ្ញាតត្រូវបានប្រើប្រាស់), ឬ​សកម្មភាព​ដែល​បាន​ធ្វើ​ប្រែចេញ​ពីក្រោម​ការ​អនុញ្ញាត​ដែល​នៅតែមានតម្លៃ (ការប្តូរប្រភេទdigests)។


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## អ្វីដែលការចងចាំសំរាប់bindingចាប់បាន

លក្ខណៈរាល់ករណីខាងក្រោមបរྟិវរណ៍ដោយ **បានបិទ** ជាមួយនឹង **ហេតុផលច្បាស់លាស់**។ ប្លុកដំបូងគឺជាការប្រមូលផ្ដុំបុរាណ (ការបំពាន, អ្នកដឹកនាំដែលច្របូកច្របល់, ការលេងឡើងវិញ, ការបោកប្រាស់នៅលើអាជ្ញាប័ណ្ណណាមួយ, បញ្ចូលព័ត៌មានមិនត្រឹមត្រូវ)។ ប្លុកទីពីរជាគូដែលធ្វើឱ្យគុណលក្ខណៈនេះពិតជាដោយហេតុ មិនមែនត្រឹមត្រូវប៉ុណ្ណោះ៖

- **អាជ្ញាប័ណ្ណចាស់** — លិខិតហត្ថលេខានៅតែមានសុពលភាព ប៉ុន្តែជំនាន់គោលនយោបាយបានផ្លាស់ប្ដូរ, កូនសោអ្នកអនុម័តត្រូវបានប្តូរចេញពីឃ្លាំងលេខធាតុដែលត្រូវបានរៀបចំ, ឬការអនុម័តបានផុតកំណត់មុនពេលបំពេញការងារ;
- **ការជំនួសមាតិកា** — លិខិតអនុម័តសកម្មភាពដែលបានហត្ថលេខាសុពលភាពដែលមាន `parent_approval_ref` ធ្វើការជូនដំណឹងទៅកាន់ការអនុម័តពិតប្រាកដមួយ ប៉ុន្តែនៅក្នុងមាតិកាសកម្មភាពទាំងមូលនៃការអនុម័តនោះ មិនត្រូវគ្នាជាមួយសកម្មភាពដែលកំពុងត្រូវបានអនុវត្តពិតប្រាកដ។


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## វត្ថុបញ្ជាក់នេះបញ្ជាក់អ្វី — ហើយវាមិនបញ្ជាក់អ្វីទេ

**បញ្ជាក់បាន:** មនុស្សដែលមានឈ្មោះអនុម័ត *សកម្មភាពកាណូនិចដូច្នេះ* (សកម្មភាពពេញលេញ + សេចក្តីសង្ខេប ដោយបានហត្ថលេខាដោយកូនសោដែលបានដោះស្រាយពីឃ្លាំងដែលត្រូវបានតំឡើង), ហើយភាគីអនុវត្ត *សកម្មភាពដែលអនុម័តនោះដូចខ្លួនវា* (សេចក្តីសង្ខេបដូចគ្នា, សក្ខីបត្រត្រូវបានភ្ជាប់ជាមួយការអនុម័តដោយ `receipt_hash`, ការតភ្ជាប់ខ្សែរបស់មេរៀន) — នៅពេលដែលកំណែគោលនយោបាយ, កូនសោ និងថ្ងៃផុតកំណត់របស់ការអនុម័តនៅតែមានសុពលភាព, ត្រឹមតែម្ដងប៉ុណ្ណោះ។ បើមានការផ្លាស់ប្តូរពីកំហុងណាមួយ ខ្សែនេះនឹងបញ្ឈប់ និងហេតុផលបដិសេធនឹងប្រាប់អ្នកថា **លក្ខណៈណាមួយ** បានខូចផ្លូវ៖ អាជ្ញាធរចាស់ vs. សកម្មភាពបានផ្លាស់ប្តូរ។

**មិនបញ្ជាក់បានទេ:** ថា UI អនុម័តបានបង្ហាញមនុស្សរឿងដែលពួកគេគិតថាកំពុងហត្ថលេខា (WYSIWYS ជាបញ្ហាដូចខ្លួនវា), ថាកូនសោមិនត្រូវបានបង្ខំឬចាប់ពីមុនការបង្វិលវិញ, ឬថាអផលបន្ទាប់បានផ្គូផ្គងនឹងសកម្មភាព។ ហត្ថលេខាដែលបានចុះហត្ថលេខា ≠ ได้รับអនុញ្ញាត: ហត្ថលេខាដែលមានសុពលភាពលើគោលនយោបាយចាស់, កូនសោបានបង្វិល, បរិវេណផុតកំណត់, ឬសេចក្តីសង្ខេបខុសគ្នានៅទីនេះមិនផ្តល់អ្វីទេ។

ប្រភេទសក្ខីបត្រពីរនេះចែករំលែកស្រោមមេរៀន និងមួយផ្លូវកូដ `verify_chain` ដោយមានគោលបំណង៖ ការតភ្ជាប់ដែលអ្នកបានបង្កើតសម្រាប់សក្ខីបត្រសកម្មភាពក្នុងសៀវភៅកំណត់ហេតុសំខាន់គឺជាកូដដូចគ្នាដែលពិនិត្យការអនុម័តនៃមនុស្ស។ កុងត្រាការត្រួតពិនិត្យមួយ, អាជ្ញាធរត្រូវបានតំឡើងបុគ្គលដាច់ដោយឡែក, ត្រូវបានបញ្ចូលដោយសេចក្តីសង្ខេបសកម្មភាពកាណូនិច ហើយគ្មានអ្វីផ្សេងទៀត។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
